# Agno Lab Exercise

Solution to the exercise at the end of Week 5 Day 3 (`5_agent_frameworks/3_maf_agno/agno_lab.ipynb`). The exercise has two parts:

1. Seed a different goal on the board, for example a short haiku about Madrid written to `madrid.txt`, and run the worker again. Does it plan sensible steps and pick the right file tools?
2. Run the agent with `await agent.aprint_response(input=..., stream=True)` instead of `arun`, and watch Agno print the tool-calling loop live as it happens.

Run the cells top to bottom with the repo's Python 3.12 kernel. The notebook is self-contained: it keeps its own board file and its own `workspace` folder in this directory, so the lab's board and workspace are untouched.

## Setup

`board.py` lives in the day 3 folder, so instead of copying it we put that folder on `sys.path` and import it from there. `BOARD_PATH` must be set before the import: it points the board at a local `board.sqlite` in this folder, which is what keeps our runs off the lab's board.

The model is built once as an `OpenAIChat` reading `OPENAI_API_KEY` from the environment, exactly as in the lab.

In [ ]:
import functools
import os
import subprocess
import sys
from pathlib import Path

# This notebook lives four levels below 5_agent_frameworks, so the day 3 folder is here:
DAY3_FOLDER = Path("../../../../3_maf_agno").resolve()
sys.path.insert(0, str(DAY3_FOLDER))

os.environ["BOARD_PATH"] = str(Path("board.sqlite").resolve())  # our own board, not the lab's

from dotenv import load_dotenv
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.mcp import MCPTools
from mcp import StdioServerParameters

import board

load_dotenv(override=True)

MODEL = "gpt-5.4-mini"
model = OpenAIChat(id=MODEL)

## The board tools, unchanged from the lab

The three board tools are copied verbatim: `show_todos` reads the board, `plan_steps` breaks a goal into steps, `complete_task` ticks one off. In Agno they are plain typed functions with no decorator; Agno reads the type hints and the docstring and builds the schema for the model.

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

## The filesystem MCP server

The same reference server as the lab, started over `npx` and scoped to a `workspace` folder inside this directory, so the agent can only touch files in there. Agno does not expose the server's stderr, so the first two lines point its stdio client at the null device, which quiets the startup banner and lets the server run from a Jupyter kernel on Windows. The server itself is described by `StdioServerParameters` and opened per run with `async with MCPTools(...)`.

In [ ]:
# Agno does not expose the MCP server's stderr, so point its stdio client at DEVNULL.
import agno.tools.mcp.mcp as agno_mcp
agno_mcp.stdio_client = functools.partial(agno_mcp.stdio_client, errlog=subprocess.DEVNULL)

workspace = Path("workspace").resolve()   # the only folder the agent may touch
workspace.mkdir(exist_ok=True)

server = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
    cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
)

## Task 1: a different goal

Seed the haiku goal and let the worker run with `arun`, exactly as the lab's step 5 did. The worker and its instruction are the same as the lab's; only the goal on the board is new.

The run is quiet until the final board: the worker should read the board, plan a couple of sensible steps under the goal, pick `write_file` to create `madrid.txt`, tick the steps off, and close the goal. Nobody tells it which file tool to use; it chooses from the MCP server's tool descriptions.

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    worker = Agent(
        model=model,
        instructions=INSTRUCTIONS,
        tools=[show_todos, plan_steps, complete_task, filesystem],
    )
    result = await worker.arun(input="Please work the pending goal on the board.")
print(result.content)

Now check the outcome: the board should show the goal and its steps struck through, and `madrid.txt` should hold the haiku.

In [ ]:
board.show_board()
print("\nmadrid.txt:\n" + (workspace / "madrid.txt").read_text(encoding="utf-8"))

## Task 2: watch the loop live with aprint_response

Same worker, same tools, one change: `await worker.aprint_response(input=..., stream=True)` instead of `arun`. Agno renders the run live as it happens: each tool call appears the moment the model makes it, followed by the streamed text of the final reply. This is Agno's window into the same loop the ADK trace, the Strands stream and the Pydantic AI transcript showed on the other days.

One practical note: the live panels draw through a Jupyter widget while the cell runs, so they do not survive into a saved copy of the notebook. Run the cell yourself to watch the stream; the board and file checks below prove the run either way.

A fresh goal gives the stream real work to show: a Lisbon haiku into a different file, so both runs' outputs sit side by side in the workspace.

In [ ]:
board.reset_board()
goal_id = board.add_goal("Write a short haiku about Lisbon into lisbon.txt.")
board.claim_todo(goal_id)

async with MCPTools(server_params=server, timeout_seconds=60) as filesystem:
    worker = Agent(
        model=model,
        instructions=INSTRUCTIONS,
        tools=[show_todos, plan_steps, complete_task, filesystem],
    )
    await worker.aprint_response(input="Please work the pending goal on the board.", stream=True)

In [ ]:
board.show_board()
print("\nlisbon.txt:\n" + (workspace / "lisbon.txt").read_text(encoding="utf-8"))